# MRI Training Pipeline

1. Configuration
2. Imports
3. Data Loading
4. Data Validation
5. Preprocessing
6. Dataset Split
7. Dataset & DataLoader
8. Model Definition
9. Training
10. Evaluation
11. Probability Output
12. Probability Calibration
13. SHAP Preparation
14. Confidence Estimation
15. Metadata Generation
16. Save Artifacts
17. Artifact Validation

In [1]:
# Parkinson's MRI Classifier (MRI Only)
# Classes: Control=0, PD=1, Prodromal=2
# Pipeline: CSV -> subject-folder mapping -> middle DICOM slice -> preprocessing ->
# Dataset/DataLoader -> ResNet18 training -> validation accuracy.

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from pathlib import Path
import random
from typing import Dict, List, Optional, Tuple
import subprocess
import sys
from copy import deepcopy

import numpy as np
import pandas as pd

try:
    import pydicom
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pydicom"])
    import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TVF
from torchvision.transforms import InterpolationMode
from PIL import Image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
import os, sys
print(sys.executable)
print(os.getcwd())

c:\Users\Sanika\anaconda3\envs\dl_gpu\python.exe
c:\Users\Sanika\OneDrive\Desktop\parkinson\models\mri


In [5]:
# Resolve project paths and prepare binary metadata config (PPMI + NTUA, MRI only)
import os

PROJECT_ROOT_OVERRIDE = r"C:/Users/Sanika/OneDrive/Desktop/parkinson"
if not PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT_OVERRIDE = os.environ.get("PARKINSON_PROJECT_ROOT", "").strip()

cwd = Path.cwd().resolve()
if str(cwd).startswith("/content"):
    raise RuntimeError(
        "Notebook is running on a remote kernel (/content), which cannot access local Windows files.\n"
        "Select local kernel and re-run from Cell 2."
    )

possible_roots = []
if PROJECT_ROOT_OVERRIDE:
    possible_roots.append(Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve())
possible_roots.append(cwd)
possible_roots.extend(list(cwd.parents))

seen = set()
unique_roots = []
for p in possible_roots:
    if str(p) not in seen:
        unique_roots.append(p)
        seen.add(str(p))

PROJECT_ROOT = None
for root in unique_roots:
    if (root / "Data").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing Data/.")

DATA_ROOT = PROJECT_ROOT / "Data"
print(f"Project root: {PROJECT_ROOT}")

# Binary labels: PD->1, Prodromal/Control->0
binary_label_map = {
    "PD": 1,
    "Prodromal": 0,
    "Control": 0,
}


def canonicalize_group(v: object) -> Optional[str]:
    if pd.isna(v):
        return None
    s = str(v).strip().lower().replace("_", "-")
    if s in {"pd", "parkinson", "parkinsons", "parkinson's"}:
        return "PD"
    if s in {"prodromal", "prodroma", "prodromal pd"}:
        return "Prodromal"
    if s in {"control", "healthy", "hc", "normal", "npd", "non-pd", "non pd"}:
        return "Control"
    return None


def find_subject_group_columns(df: pd.DataFrame) -> Tuple[str, str]:
    subject_candidates = ["Subject", "subject", "SUBJECT", "PatientID", "Patient_ID", "ID", "id", "Alias", "alias"]
    group_candidates = ["Group", "group", "Diagnosis", "diagnosis", "Label", "label", "Class", "class", "DIAGN"]

    subject_col = next((c for c in subject_candidates if c in df.columns), None)
    group_col = next((c for c in group_candidates if c in df.columns), None)

    if subject_col is None or group_col is None:
        raise ValueError(f"Could not find Subject/Group columns in: {list(df.columns)}")
    return subject_col, group_col


# 1) PPMI source (DICOM)
ppmi_csv = DATA_ROOT / "MRI_4_10_2026.csv"
ppmi_mri_root = DATA_ROOT / "PPMI" / "mri"
if not ppmi_csv.exists():
    raise FileNotFoundError(f"PPMI CSV not found: {ppmi_csv}")
if not ppmi_mri_root.exists():
    raise FileNotFoundError(f"PPMI MRI folder not found: {ppmi_mri_root}")

raw = pd.read_csv(ppmi_csv)
subject_col, group_col = find_subject_group_columns(raw)

ppmi_meta = raw[[subject_col, group_col]].copy()
ppmi_meta.columns = ["Subject", "Group"]
ppmi_meta["Group"] = ppmi_meta["Group"].apply(canonicalize_group)
ppmi_meta = ppmi_meta.dropna(subset=["Subject", "Group"]).reset_index(drop=True)
ppmi_meta["label"] = ppmi_meta["Group"].map(binary_label_map)
ppmi_meta = ppmi_meta.dropna(subset=["label"]).copy()
ppmi_meta["label"] = ppmi_meta["label"].astype(int)

print(f"[PPMI] CSV: {ppmi_csv}")
print(f"[PPMI] MRI root: {ppmi_mri_root}")
print(f"[PPMI] rows: {len(ppmi_meta)}")

# 2) NTUA source (image files)
ntua_candidates = [
    DATA_ROOT / "ntua-parkinson-dataset-master",
    DATA_ROOT / "PPMI" / "ntua-parkinson-dataset-master",
]
NTUA_ROOT = next((p for p in ntua_candidates if p.exists()), None)
if NTUA_ROOT is None:
    raise FileNotFoundError("NTUA folder not found in Data/.")

pd_patients = NTUA_ROOT / "PD Patients"
npd_patients = NTUA_ROOT / "Non PD Patients"
if not pd_patients.exists() or not npd_patients.exists():
    raise FileNotFoundError(f"NTUA PD/Non PD folders not found under: {NTUA_ROOT}")

ntua_rows: List[Dict[str, object]] = []
for sub in pd_patients.iterdir():
    if sub.is_dir() and sub.name.lower().startswith("subject"):
        ntua_rows.append({"Subject": sub.name, "Group": "PD", "label": 1})
for sub in npd_patients.iterdir():
    if sub.is_dir() and sub.name.lower().startswith("subject"):
        ntua_rows.append({"Subject": sub.name, "Group": "Control", "label": 0})

ntua_meta = pd.DataFrame(ntua_rows)
print(f"[NTUA] root: {NTUA_ROOT}")
print(f"[NTUA] rows: {len(ntua_meta)}")

dataset_tables = [
    {"source": "PPMI", "source_type": "dcm", "mri_root": ppmi_mri_root, "meta": ppmi_meta},
    {"source": "NTUA", "source_type": "img", "mri_root": NTUA_ROOT, "meta": ntua_meta},
]
print(f"Loaded dataset sources: {[d['source'] for d in dataset_tables]}")

Project root: C:\Users\Sanika\OneDrive\Desktop\parkinson
[PPMI] CSV: C:\Users\Sanika\OneDrive\Desktop\parkinson\Data\MRI_4_10_2026.csv
[PPMI] MRI root: C:\Users\Sanika\OneDrive\Desktop\parkinson\Data\PPMI\mri
[PPMI] rows: 104
[NTUA] root: C:\Users\Sanika\OneDrive\Desktop\parkinson\Data\PPMI\ntua-parkinson-dataset-master
[NTUA] rows: 78
Loaded dataset sources: ['PPMI', 'NTUA']


In [6]:
# Map Subject -> folder for each source, then combine into one unified binary table
if "dataset_tables" not in globals() or len(dataset_tables) == 0:
    raise RuntimeError("dataset_tables is empty. Run Cell 5 first.")


def normalize_subject_id(value: object) -> Optional[str]:
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits:
        return str(int(digits))
    return s.lower() if s else None


def leading_digits(text: str) -> str:
    out = []
    for ch in text:
        if ch.isdigit():
            out.append(ch)
        else:
            break
    return "".join(out)


def build_subject_folder_index(mri_root: Path) -> Dict[str, Path]:
    index: Dict[str, Path] = {}
    for subdir in mri_root.iterdir():
        if not subdir.is_dir():
            continue
        name = subdir.name
        all_digits = "".join(ch for ch in name if ch.isdigit())
        lead_digits = leading_digits(name)
        keys = {name.lower()}
        for d in [all_digits, lead_digits]:
            if d:
                keys.add(d)
                keys.add(str(int(d)))
        for k in keys:
            index.setdefault(k, subdir)
    return index


mapped_parts: List[pd.DataFrame] = []
for item in dataset_tables:
    source = item["source"]
    source_type = item["source_type"]
    src_root = item["mri_root"]
    src_meta = item["meta"].copy()

    src_meta["source"] = source
    src_meta["source_type"] = source_type

    if source_type == "dcm":
        subject_index = build_subject_folder_index(Path(src_root))
        src_meta["subject_key"] = src_meta["Subject"].apply(normalize_subject_id)
        src_meta["subject_folder"] = src_meta["subject_key"].apply(
            lambda x: subject_index.get(x) if x is not None else None
        )
    else:
        pd_root = Path(src_root) / "PD Patients"
        npd_root = Path(src_root) / "Non PD Patients"

        def ntua_subject_folder(subject_name: object, label: int) -> Optional[Path]:
            name = str(subject_name).strip()
            base = pd_root if int(label) == 1 else npd_root
            cand = base / name / "1.MRI"
            if cand.exists():
                return cand
            cand2 = base / name
            if cand2.exists():
                return cand2
            return None

        src_meta["subject_key"] = src_meta["Subject"].astype(str)
        src_meta["subject_folder"] = src_meta.apply(
            lambda r: ntua_subject_folder(r["Subject"], r["label"]), axis=1
        )

    mapped = src_meta["subject_folder"].notna().sum()
    print(f"[{source}] mapped rows: {mapped}/{len(src_meta)}")
    mapped_parts.append(src_meta)

meta = pd.concat(mapped_parts, ignore_index=True)
print(f"Unified rows: {len(meta)}")
print("Unified label counts (0=Non-PD,1=PD):")
print(meta["label"].value_counts().sort_index())
meta.head()

[PPMI] mapped rows: 100/104
[NTUA] mapped rows: 78/78
Unified rows: 182
Unified label counts (0=Non-PD,1=PD):
label
0     59
1    123
Name: count, dtype: int64


,Subject,Group,label,source,source_type,subject_key,subject_folder
0,75426,Prodromal,0,PPMI,dcm,75426,C:\Users\Sanika\OneDrive\Desktop\parkinson\Dat...
1,75409,PD,1,PPMI,dcm,75409,C:\Users\Sanika\OneDrive\Desktop\parkinson\Dat...
2,74739,Prodromal,0,PPMI,dcm,74739,C:\Users\Sanika\OneDrive\Desktop\parkinson\Dat...
3,74312,Prodromal,0,PPMI,dcm,74312,C:\Users\Sanika\OneDrive\Desktop\parkinson\Dat...
4,74307,Prodromal,0,PPMI,dcm,74307,C:\Users\Sanika\OneDrive\Desktop\parkinson\Dat...


In [7]:
def find_dicom_files(subject_folder: Path) -> List[Path]:
    files = list(subject_folder.rglob("*.dcm")) + list(subject_folder.rglob("*.DCM"))
    return sorted(set(files))


def find_image_files(subject_folder: Path) -> List[Path]:
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    files: List[Path] = []
    for ext in exts:
        files.extend(subject_folder.rglob(ext))
        files.extend(subject_folder.rglob(ext.upper()))
    return sorted(set(files))


def _slice_sort_key(ds: pydicom.Dataset, fallback: int) -> float:
    if hasattr(ds, "InstanceNumber"):
        try:
            return float(ds.InstanceNumber)
        except Exception:
            pass
    if hasattr(ds, "ImagePositionPatient"):
        try:
            return float(ds.ImagePositionPatient[2])
        except Exception:
            pass
    return float(fallback)


def select_central_indices(n: int, k: int = 7) -> List[int]:
    if n <= 0:
        return []
    k = max(1, min(k, n))
    center = n // 2
    start = center - (k // 2)
    if start < 0:
        start = 0
    end = start + k
    if end > n:
        end = n
        start = end - k
    return list(range(start, end))


def collect_ppmi_central_slice_files(subject_folder: Path, num_central_slices: int = 7) -> List[Path]:
    dicom_paths = find_dicom_files(subject_folder)
    if not dicom_paths:
        return []

    keyed: List[Tuple[float, Path]] = []
    for i, p in enumerate(dicom_paths):
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
            keyed.append((_slice_sort_key(ds, i), p))
        except Exception:
            continue

    if not keyed:
        return []

    keyed.sort(key=lambda x: x[0])
    ordered_paths = [x[1] for x in keyed]
    idxs = select_central_indices(len(ordered_paths), k=num_central_slices)
    return [ordered_paths[i] for i in idxs]


def load_dicom_slice_from_file(dcm_path: Path) -> Optional[np.ndarray]:
    try:
        ds = pydicom.dcmread(dcm_path)
        arr = ds.pixel_array.astype(np.float32)
        if arr.ndim != 2:
            return None
        if not np.isfinite(arr).all():
            return None
        if float(arr.max()) <= float(arr.min()):
            return None
        return arr
    except Exception:
        return None


def load_image_slice_from_file(img_path: Path) -> Optional[np.ndarray]:
    try:
        arr = np.array(Image.open(img_path).convert("L"), dtype=np.float32)
        if arr.ndim != 2:
            return None
        if not np.isfinite(arr).all():
            return None
        if float(arr.max()) <= float(arr.min()):
            return None
        return arr
    except Exception:
        return None


def preprocess_to_3ch_tensor(img_2d: np.ndarray, size: int = 224) -> torch.Tensor:
    if img_2d.ndim != 2:
        raise ValueError(f"Expected 2D image, got shape: {img_2d.shape}")

    img = img_2d.astype(np.float32)
    min_v, max_v = float(img.min()), float(img.max())
    if max_v > min_v:
        img = (img - min_v) / (max_v - min_v)
    else:
        img = np.zeros_like(img, dtype=np.float32)

    x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # [1,1,H,W]
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    x = x.squeeze(0).repeat(3, 1, 1)  # [3,224,224]
    return x

In [8]:
class MRIBinaryDataset(Dataset):
    def __init__(
        self,
        samples: List[Tuple[Path, int, str]],
        indices: List[int],
        augment: bool = False,
        max_rotate_deg: float = 8.0,
    ):
        self.samples = samples
        self.indices = indices
        self.augment = augment
        self.max_rotate_deg = max_rotate_deg

    def __len__(self) -> int:
        return len(self.indices)

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() < 0.5:
            x = torch.flip(x, dims=[2])
        angle = random.uniform(-self.max_rotate_deg, self.max_rotate_deg)
        x = TVF.rotate(x, angle=angle, interpolation=InterpolationMode.BILINEAR)
        return x

    def __getitem__(self, idx: int):
        real_idx = self.indices[idx]
        file_path, label, source_type = self.samples[real_idx]

        if source_type == "dcm_slice":
            img = load_dicom_slice_from_file(file_path)
        elif source_type == "img_file":
            img = load_image_slice_from_file(file_path)
        else:
            raise ValueError(f"Unknown source_type: {source_type}")

        if img is None:
            x = torch.zeros((3, 224, 224), dtype=torch.float32)
        else:
            x = preprocess_to_3ch_tensor(img, size=224)

        if self.augment:
            x = self._augment(x)

        y = torch.tensor(label, dtype=torch.long)
        return x, y


# Build slice-level/sample-level list from combined metadata
# PPMI: central N DICOM slices per subject folder -> each slice is a sample
# NTUA: each image file is a sample
all_samples: List[Tuple[Path, int, str]] = []
skipped_missing_folder = 0
ppmi_slice_samples = 0
ntua_image_samples = 0

central_slices_per_subject = 7  # use 5-10 recommended central slices

for _, row in meta.iterrows():
    folder = row["subject_folder"]
    label = int(row["label"])
    source_type = str(row["source_type"])

    if folder is None or not Path(folder).exists():
        skipped_missing_folder += 1
        continue

    folder = Path(folder)

    if source_type == "dcm":
        selected_dcm = collect_ppmi_central_slice_files(folder, num_central_slices=central_slices_per_subject)
        for dcm_path in selected_dcm:
            all_samples.append((dcm_path, label, "dcm_slice"))
            ppmi_slice_samples += 1

    elif source_type == "img":
        img_files = find_image_files(folder)
        for img_path in img_files:
            all_samples.append((img_path, label, "img_file"))
            ntua_image_samples += 1

if len(all_samples) == 0:
    raise RuntimeError("No valid MRI slice/image samples found.")

print(f"Total valid samples (slice-level): {len(all_samples)}")
print(f"PPMI central DICOM slice samples: {ppmi_slice_samples}")
print(f"NTUA image-file samples: {ntua_image_samples}")
print(f"Skipped missing-folder rows: {skipped_missing_folder}")

all_labels = np.array([y for _, y, _ in all_samples], dtype=np.int64)
print(
    "Overall class counts (0=Non-PD,1=PD):",
    {0: int((all_labels == 0).sum()), 1: int((all_labels == 1).sum())},
)


# Train/Validation split
def stratified_train_val_indices(labels: np.ndarray, train_ratio: float = 0.8, seed: int = 42):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []

    for c in np.unique(labels):
        cls_idx = np.where(labels == c)[0]
        rng.shuffle(cls_idx)
        n_train = max(1, int(round(len(cls_idx) * train_ratio)))
        if n_train >= len(cls_idx) and len(cls_idx) > 1:
            n_train = len(cls_idx) - 1
        train_idx.extend(cls_idx[:n_train].tolist())
        val_idx.extend(cls_idx[n_train:].tolist())

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


train_idx, val_idx = stratified_train_val_indices(all_labels, train_ratio=0.8, seed=SEED)

train_ds = MRIBinaryDataset(all_samples, train_idx, augment=True, max_rotate_deg=8.0)
val_ds = MRIBinaryDataset(all_samples, val_idx, augment=False)

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

train_labels = np.array([all_samples[i][1] for i in train_idx], dtype=np.int64)
train_class_counts = np.bincount(train_labels, minlength=2)

inv_sqrt = np.where(train_class_counts > 0, 1.0 / np.sqrt(train_class_counts), 0.0)
if (inv_sqrt > 0).any():
    inv_sqrt = inv_sqrt / inv_sqrt[inv_sqrt > 0].mean()
train_class_weights_np = np.clip(inv_sqrt, 0.5, 3.0)

sample_weights = train_class_weights_np[train_labels]
train_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

batch_size = 4  # keep small for slice-level dataset
train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

class_weights = torch.tensor(train_class_weights_np, dtype=torch.float32)

print("Train class counts [Non-PD, PD]:", train_class_counts.tolist())
print("Train loss weights [Non-PD, PD]:", class_weights.tolist())

val_labels = np.array([all_samples[i][1] for i in val_idx], dtype=np.int64) if len(val_idx) else np.array([], dtype=np.int64)
print("Val class counts [Non-PD, PD]:", np.bincount(val_labels, minlength=2).tolist() if len(val_labels) else [0, 0])

Total valid samples (slice-level): 41257
PPMI central DICOM slice samples: 700
NTUA image-file samples: 40557
Skipped missing-folder rows: 4
Overall class counts (0=Non-PD,1=PD): {0: 10170, 1: 31087}
Train size: 33006, Val size: 8251
Train class counts [Non-PD, PD]: [8136, 24870]
Train loss weights [Non-PD, PD]: [1.2722949981689453, 0.7277050614356995]
Val class counts [Non-PD, PD]: [2034, 6217]


In [9]:
# Verify sample shapes
x_batch, y_batch = next(iter(train_loader))
print("Batch image shape:", x_batch.shape)  # [B, 3, 224, 224]
print("Batch label shape:", y_batch.shape)
print("Unique labels in this batch:", y_batch.unique().tolist())

Batch image shape: torch.Size([4, 3, 224, 224])
Batch label shape: torch.Size([4])
Unique labels in this batch: [0, 1]


In [10]:
# Build pretrained ResNet18 for binary classification
try:
    weights = ResNet18_Weights.DEFAULT
except Exception:
    weights = None

model = resnet18(weights=weights)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Model ready:", model.__class__.__name__)

Model ready: ResNet


In [11]:
# Training loop (5-10 epochs requested)
num_epochs = 5
best_val_acc = -1.0
best_state = deepcopy(model.state_dict())

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = running_loss / max(1, train_total)
    train_acc = train_correct / max(1, train_total)

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / max(1, val_total)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = deepcopy(model.state_dict())

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}"
    )

model.load_state_dict(best_state)
print(f"Best Val Accuracy: {best_val_acc:.4f}")

Epoch [1/5] | Train Loss: 0.4688 | Train Acc: 0.7741 | Val Acc: 0.8937
Epoch [2/5] | Train Loss: 0.2356 | Train Acc: 0.9034 | Val Acc: 0.9353
Epoch [3/5] | Train Loss: 0.1587 | Train Acc: 0.9382 | Val Acc: 0.9452
Epoch [4/5] | Train Loss: 0.1206 | Train Acc: 0.9540 | Val Acc: 0.9547
Epoch [5/5] | Train Loss: 0.0965 | Train Acc: 0.9636 | Val Acc: 0.9663
Best Val Accuracy: 0.9663


In [12]:
# Evaluation: print validation accuracy
model.eval()
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(images)
        preds = logits.argmax(dim=1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

val_acc = val_correct / max(1, val_total)
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.9663


In [13]:
import pickle

ARTIFACT_PKL = PROJECT_ROOT / "mri_model_artifact.pkl"

artifact = {
    "model_name": "resnet18",
    "num_classes": 2,
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "class_names": {0: "Non-PD", 1: "PD"},
    "seed": SEED,
    "train_indices": train_idx,
    "val_indices": val_idx,
    "best_val_acc": best_val_acc,
    "input_size": 224,
}

with open(ARTIFACT_PKL, "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved pickle artifact to: {ARTIFACT_PKL}")


Saved pickle artifact to: C:\Users\Sanika\OneDrive\Desktop\parkinson\mri_model_artifact.pkl


### Optional tuning, SHAP preparation, and uncertainty metadata

The model below already has a reasonable baseline configuration. This section records the selected hyperparameters, saves a SHAP reference batch for inference-time explainability, and keeps the uncertainty contract explicit in metadata.

In [14]:
# Optional tuning record. The current notebook uses a strong default configuration,
# so we keep the selected learning rate explicit and avoid an expensive search by default.
RUN_HYPERPARAMETER_TUNING = False
candidate_learning_rates = [1e-4, 5e-5, 1e-5]
selected_learning_rate = 1e-4
selected_tuning_result = {
    "selected_learning_rate": selected_learning_rate,
    "tuning_enabled": RUN_HYPERPARAMETER_TUNING,
    "candidate_learning_rates": candidate_learning_rates,
    "reason": "Baseline configuration already trained in this notebook.",
}

artifact_dir = PROJECT_ROOT / "artifacts" / "mri"
artifact_dir.mkdir(parents=True, exist_ok=True)

if RUN_HYPERPARAMETER_TUNING:
    tuning_results = []
    for learning_rate in candidate_learning_rates:
        trial_model = resnet18(weights=weights)
        trial_model.fc = nn.Linear(trial_model.fc.in_features, 2)
        trial_model = trial_model.to(DEVICE)
        trial_optimizer = torch.optim.Adam(trial_model.parameters(), lr=learning_rate)
        trial_model.train()
        trial_images, trial_labels = next(iter(train_loader))
        trial_images = trial_images.to(DEVICE)
        trial_labels = trial_labels.to(DEVICE)
        trial_optimizer.zero_grad()
        trial_logits = trial_model(trial_images)
        trial_loss = criterion(trial_logits, trial_labels)
        trial_loss.backward()
        trial_optimizer.step()
        trial_model.eval()
        trial_correct = 0
        trial_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                logits = trial_model(images)
                preds = logits.argmax(dim=1)
                trial_correct += (preds == labels).sum().item()
                trial_total += labels.size(0)
        tuning_results.append({"learning_rate": learning_rate, "val_accuracy": trial_correct / max(1, trial_total)})
    selected_tuning_result = max(tuning_results, key=lambda row: row["val_accuracy"])
    selected_learning_rate = selected_tuning_result["learning_rate"]

print("Selected tuning result:", selected_tuning_result)

# Save a compact SHAP reference batch and the associated labels.
shap_reference_images, shap_reference_labels = next(iter(train_loader))
shap_reference_artifact = {
    "reference_images": shap_reference_images[:8].cpu(),
    "reference_labels": shap_reference_labels[:8].cpu(),
    "feature_names": ["central_dicom_slice_rgb_224x224"],
    "shap_backend": "captum",
    "explanation_method": "GradientShap_or_IntegratedGradients",
}
torch.save(shap_reference_artifact, artifact_dir / "shap_reference_batch.pt")

TUNING_INFO = {
    "selected_learning_rate": selected_learning_rate,
    "tuning_enabled": RUN_HYPERPARAMETER_TUNING,
    "candidate_learning_rates": candidate_learning_rates,
    "selected_tuning_result": selected_tuning_result,
    "shap_artifact": "shap_reference_batch.pt",
    "shap_backend": "captum",
    "supports_uncertainty": False,
    "uncertainty_method": None,
}


Selected tuning result: {'selected_learning_rate': 0.0001, 'tuning_enabled': False, 'candidate_learning_rates': [0.0001, 5e-05, 1e-05], 'reason': 'Baseline configuration already trained in this notebook.'}


### Export artifacts, calibration, and inference validation

This final section saves the trained model and the inference-time artifacts required by deployment, then reloads them and runs one prediction to catch serialization issues early.

In [15]:
from datetime import datetime, timezone
import json
import pickle

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "mri"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

preprocess_config = {
    "resize": [224, 224],
    "channels": 3,
    "normalization": {"type": "min_max_per_slice"},
    "steps": [
        "load DICOM slice",
        "validate finite pixels",
        "scale to [0, 1] per slice",
        "resize to 224x224",
        "repeat grayscale channel to RGB",
    ],
}
with open(ARTIFACT_DIR / "preprocess_config.json", "w", encoding="utf-8") as handle:
    json.dump(preprocess_config, handle, indent=2)


def collect_model_outputs(loader):
    model.eval()
    all_probs = []
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            all_preds.extend(preds.detach().cpu().numpy().tolist())
    return np.asarray(all_probs, dtype=np.float32), np.asarray(all_labels, dtype=np.int64), np.asarray(all_preds, dtype=np.int64)


val_probs, val_labels, val_preds = collect_model_outputs(val_loader)

metrics = {
    "accuracy": float(accuracy_score(val_labels, val_preds)),
    "precision": float(precision_score(val_labels, val_preds, zero_division=0)),
    "recall": float(recall_score(val_labels, val_preds, zero_division=0)),
    "f1": float(f1_score(val_labels, val_preds, zero_division=0)),
    "roc_auc": float(roc_auc_score(val_labels, val_probs)) if len(np.unique(val_labels)) > 1 else None,
    "pr_auc": float(average_precision_score(val_labels, val_probs)) if len(np.unique(val_labels)) > 1 else None,
    "confusion_matrix": confusion_matrix(val_labels, val_preds).tolist(),
}

print("MRI validation metrics:")
for name, value in metrics.items():
    print(f"{name}: {value}")

calibrator = None
if len(np.unique(val_probs)) > 1:
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(val_probs, val_labels)

model_state = {
    "model_name": "resnet18",
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "class_names": {0: "Non-PD", 1: "PD"},
    "input_size": 224,
    "seed": SEED,
    "modality": "mri",
}

torch.save(model_state, ARTIFACT_DIR / "model.pt")
with open(ARTIFACT_DIR / "calibrator.pkl", "wb") as handle:
    pickle.dump(calibrator, handle)

metadata = {
    "model_id": "mri_resnet18_binary",
    "modality": "mri",
    "architecture": "resnet18",
    "validation_accuracy": metrics["accuracy"],
    "validation_precision": metrics["precision"],
    "validation_recall": metrics["recall"],
    "validation_f1": metrics["f1"],
    "validation_auc": metrics["roc_auc"],
    "validation_pr_auc": metrics["pr_auc"],
    "feature_names": ["central_dicom_slice_rgb_224x224"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_version": "v1",
    "supports_uncertainty": False,
    "supports_shap": True,
    "supports_calibration": True,
    "mc_samples": 0,
    "shap_backend": "captum",
    "shap_artifact": "shap_reference_batch.pt",
    "selected_learning_rate": TUNING_INFO["selected_learning_rate"] if "TUNING_INFO" in globals() else 1e-4,
    "candidate_learning_rates": TUNING_INFO["candidate_learning_rates"] if "TUNING_INFO" in globals() else [1e-4, 5e-5, 1e-5],
    "tuning_enabled": TUNING_INFO["tuning_enabled"] if "TUNING_INFO" in globals() else False,
    "artifacts": {
        "model": "model.pt",
        "calibrator": "calibrator.pkl",
        "preprocess_config": "preprocess_config.json",
        "shap_reference": "shap_reference_batch.pt",
    },
}

with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

legacy_artifact_path = PROJECT_ROOT / "mri_model_artifact.pkl"
with open(legacy_artifact_path, "wb") as handle:
    pickle.dump({**model_state, "metrics": metrics, "metadata": metadata}, handle)

print(f"Saved MRI artifacts to: {ARTIFACT_DIR}")
print(f"Saved legacy artifact to: {legacy_artifact_path}")

# Inference validation: reload the model and run one prediction.
reloaded = resnet18(weights=None)
reloaded.fc = nn.Linear(reloaded.fc.in_features, 2)
reloaded.load_state_dict(model_state["state_dict"])
reloaded = reloaded.to(DEVICE)
reloaded.eval()

sample_x, sample_y = val_ds[0]
with torch.no_grad():
    sample_logits = reloaded(sample_x.unsqueeze(0).to(DEVICE))
    sample_prob = torch.softmax(sample_logits, dim=1)[0, 1].item()
    sample_pred = int(sample_logits.argmax(dim=1).item())

print({
    "sample_true_label": int(sample_y.item()),
    "sample_pred_label": sample_pred,
    "sample_pd_probability": float(sample_prob),
})

MRI validation metrics:
accuracy: 0.966307114289177
precision: 0.9799579764021336
recall: 0.9752292102300145
f1: 0.9775878748790713
roc_auc: 0.9926475112092339
pr_auc: 0.9974406153735618
confusion_matrix: [[1910, 124], [154, 6063]]
Saved MRI artifacts to: C:\Users\Sanika\OneDrive\Desktop\parkinson\artifacts\mri
Saved legacy artifact to: C:\Users\Sanika\OneDrive\Desktop\parkinson\mri_model_artifact.pkl
{'sample_true_label': 1, 'sample_pred_label': 1, 'sample_pd_probability': 0.9999434947967529}
